<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/mates/notebooks/c3_l4.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C3-L4 · Volatilidad y ATR
Vol realizada anualizada, True Range, ATR(14) y stops k×ATR con OHLC de BNB.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/mates/data/c3_l4.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c3_l4.csv'), Path('data/c3_l4.csv'), Path('c3_l4.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)

In [ ]:
df['ret_log'] = np.log(df['close'] / df['close'].shift(1))
vol_d = df['ret_log'].std(ddof=1)
print(f'vol_diaria={vol_d:.4f}  vol_anual_x365={vol_d*np.sqrt(365):.2%}')

In [ ]:
prev = df['close'].shift(1)
df['tr'] = np.maximum(df['high']-df['low'], np.maximum((df['high']-prev).abs(), (df['low']-prev).abs()))
df['tr'] = df['tr'].fillna(df['high']-df['low'])  # fila 1: sin close previo
df['atr14'] = df['tr'].rolling(14).mean()
print(df[['dia','close','tr','atr14']].tail(8).to_string(index=False))

In [ ]:
for k in [1.0, 2.0, 3.0]:
    df[f'stop{k:g}'] = df['close'] - k*df['atr14']
    tocados = int((df['low'] < df[f'stop{k:g}']).sum())
    print(f'k={k:g}: dias_con_min_bajo_stop={tocados}/50')

In [ ]:
# Chequeo automático
assert (df['tr'] >= (df['high']-df['low']) - 1e-9).all(), 'TR >= high-low'
assert df['atr14'].iloc[13] == df['tr'].iloc[:14].mean()
assert (df['stop2'].dropna() < df.loc[df['stop2'].notna(), 'close']).all(), 'stop bajo el precio'
print('OK: ATR y stops verificados')